In [132]:
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib
import numpy as np
import matplotlib.pyplot as plt
import joblib
import anndata as ad
from scipy.signal import find_peaks

In [17]:
test = sc.read("C:\\Users\\olesk\\Downloads\\local.h5ad")

In [18]:
test

AnnData object with n_obs × n_vars = 23686 × 36503
    obs: 'donor_id', 'self_reported_ethnicity_ontology_term_id', 'organism_ontology_term_id', 'sample_uuid', 'sample_preservation_method', 'tissue_ontology_term_id', 'development_stage_ontology_term_id', 'donor_BMI_at_collection', 'tissue_section_uuid', 'tissue_section_thickness', 'library_uuid', 'assay_ontology_term_id', 'mapped_reference_annotation', 'is_primary_data', 'cell_type_ontology_term_id', 'author_cell_type', 'disease_ontology_term_id', 'sex_ontology_term_id', 'nCount_Spatial', 'nFeature_Spatial', 'nCount_SCT', 'nFeature_SCT', 'suspension_type', 'cell_type', 'assay', 'disease', 'organism', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage'
    var: 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype'
    uns: 'default_embedding', 'schema_version', 'title'
    obsm: 'X_pca', 'X_umap'

In [133]:
prot = joblib.load("C:\\Users\\olesk\\ECM_mod_output\\UMAP\\all_prot_0515.sav")

In [155]:
prot.shape

(149886, 1440)

In [134]:
umap_embed = joblib.load("C:\\Users\\olesk\\ECM_mod_output\\UMAP\\embedding_0515.sav")

In [135]:
adata = ad.AnnData(prot)

C:\Users\olesk\anaconda3\envs\cellxgene\lib\site-packages\ipykernel_launcher.py:1: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  """Entry point for launching an IPython kernel.


In [136]:
adata.obs_names = [f"win_{i:d}" for i in range(adata.n_obs)]

In [138]:
prot_names = ["skp2", "p21", "pp21", "cdh1", "myc", "cycD", "cycD-p21", 
        "Rb", "pRb", "E2F", "E2F-Rb", "cycE", "cycE-p21", "cycA", "cycA-p21",
        "cycB", "cycB-p21", "cdc20"]

all_names = []
for name in prot_names:
    tmp_name = [f"{name}_{i:d}" for i in range(80)]
    all_names = all_names + tmp_name
    
adata.var_names = all_names

In [139]:
dt = ['DMSO']*74943 + ['ERL']*74943
adata.obs["drug"] = pd.Categorical(dt)

In [140]:
st = [0.2]*24981 + [4]*24981 + [55]*24981 + [0.2]*24981 + [4]*24981 + [55]*24981
adata.obs["stiffness"] = pd.Categorical(st)

In [141]:
adata.obsm["X_umap"] = umap_embed

In [210]:
def getIMT(prot):
    peaks_E, _ = find_peaks(prot[:,int((80*11+80/2)-1)], prominence = 40/200, distance = 10)
    peaks_cdc, _ = find_peaks(prot[:,int((80*17+80/2)-1)], prominence = 40/200, distance = 10)
    
    time_st = 0.2
    
    tmp = np.empty((peaks_E[3],1))
    tmp[:] = np.nan
    div_time = pd.DataFrame(tmp)
    
    for n in range(3,len(peaks_E)-3):   #make sure we have divisions at ends 
        tmp_g1_s = peaks_E[n]
        tmp_g1_s_2 = peaks_E[n+1]

        tmp_mit = peaks_cdc[(tmp_g1_s < peaks_cdc) & (peaks_cdc < tmp_g1_s_2)]
          
             
        if np.size(tmp_mit) > 0:
            ordered = [tmp_g1_s, tmp_mit[0], tmp_g1_s_2]
            g1_len_tmp = np.array([ordered[2]*time_st]) - np.array([ordered[1]*time_st])
            tmp = [g1_len_tmp]*int(ordered[2]-ordered[0])
            tmp = np.concatenate(tmp, axis=0)
            tmp_df = pd.DataFrame(tmp)

            div_time = pd.concat([div_time, tmp_df], ignore_index=True)
        else:
            tmp = np.empty((int(peaks_E[n+1]-peaks_E[n]),1))
            tmp[:] = np.nan
            tmp_df = pd.DataFrame(tmp)
            div_time = pd.concat([div_time, tmp_df], ignore_index=True) 
    
    
    tmp = np.empty((24981-peaks_E[-3],1))
    tmp[:] = np.nan
    tmp_df = pd.DataFrame(tmp)
    div_time = pd.concat([div_time, tmp_df], ignore_index=True) 
        
    return div_time

In [193]:
test = prot[0*24981:1*24981,:]


(24981, 1440)

In [113]:
tmp = np.empty((int(peaks_cdc[n+1]-peaks_cdc[n]),1))
tmp[:] = np.nan
tmp_df = pd.DataFrame(tmp)

In [120]:
(np.size(tmp_g1_s) > 0) & (np.size(tmp_g1_s_2)>0) & (np.size(tmp_g1_s_3)>0)

True

In [241]:
test_t = getIMT(prot[1*24981:2*24981,:])

In [232]:
tmp = test_t[0].values.tolist()
len(tmp)

24981

In [242]:
all_divs = []
for num in range(6):
    tmp_div = getIMT(prot[num*24981:(num+1)*24981,:])
    tmp_list = tmp_div[0].values.tolist()
    all_divs = all_divs + tmp_list
    print(num)

0
1
2
3
4
5


In [274]:
G1_cats = []
for val in all_divs:
    if val <= 15:
        tmp = 'Short'
    elif (val > 15) & (val <= 25):
        tmp = 'Int'
    elif val > 25:
        tmp = 'Long'
    else:
        tmp = 'nan'
    
    G1_cats.append(tmp)

In [275]:
len(G1_cats)

149886

In [276]:
G1_cats

['nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',


In [277]:
adata.obs["G1_time"] = pd.Categorical(G1_cats)

In [278]:
adata.write('my_results.h5ad')